# Homework: Learning Audio Representations with Self-Supervision

In this homework, we will consolidate the knowledge gained during the seminar and further explore methods for self-supervised learning of audio representations.
During the seminar, we implemented and trained a model based on contrastive learning (InfoNCE). Now, we will extend this work by implementing non-contrastive learning (NCL) approaches and comparing them with previously studied methods.

We will examine how different training paradigms — supervised, contrastive, and non-contrastive — affect the quality of learned embeddings and the stability of the training process


## Multi-Format Contrastive Learning of Audio Representations(Paper from seminar)

The core idea of this approach is to learn robust audio embeddings by contrasting
different *formats* (or views) of the same audio sample.  
For example, one branch may encode the **raw waveform (1D)** while another encodes
its **spectrogram (2D)**.  

By applying a contrastive loss (InfoNCE), the model is trained to:
- **Pull together** embeddings from different formats of the same audio,
- **Push apart** embeddings from different audio samples.  

This multi-format setup encourages the encoder to capture **shared semantic content**
across input representations, leading to more general and transferable audio features.

[paper1](https://arxiv.org/pdf/2103.06508), [paper2](https://arxiv.org/pdf/2010.09542)

[github source](https://github.com/HondamunigePrasannaSilva/CLAR?tab=readme-ov-file)

#### Load libraries

In [1]:
import os
import random
import torchaudio
import torchaudio.transforms as T
import matplotlib.pyplot as plt
from IPython.display import Audio
from pathlib import Path
from omegaconf import DictConfig

import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from tqdm.notebook import tqdm
from torch.utils.data import Subset, DataLoader
from torchaudio import transforms
from sklearn.manifold import TSNE

import warnings
warnings.filterwarnings("ignore", module="torchaudio._backend")

# plt.rcParams.update({'font.size': 14})

#### AudioSet

Dataset with 10 numbers pronounced
```
!git clone https://github.com/soerenab/AudioMNIST.git
```



In [2]:
!git clone https://github.com/soerenab/AudioMNIST.git

fatal: destination path 'AudioMNIST' already exists and is not an empty directory.


## Task 1(2 points)

Impelment strightforward classifier training based on waveforms or spectrogram(use models from seminar). Train it in supervised manner and compute accuracy

#### Подготовка данных

In [3]:
root = './AudioMNIST/data'
# root = '/content/AudioMNIST/data'
#check for cuda
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Available device:", device)

Available device: cuda


##### Объявим кастомный Dataset класс **AudioMNISTDataset** и функцию **collate**

In [4]:

class AudioMNISTDataset(torch.utils.data.Dataset):
    def __init__(self, root, augmentation=False, sr=16000):
        self.root = root
        self.augmentation = augmentation
        self.sr = sr
        self.items = self.list_wavs_and_labels(root)

    def __len__(self):
      return len(self.items)

    def list_wavs_and_labels(self, root: str):
      base = Path(root)
      speakers = sorted([p for p in base.iterdir() if p.is_dir()])
      items = []
      for sp in speakers:
          for wav in sorted(sp.glob("**/*.wav")):
              # filename e.g., "9_10_0_0_1.wav" (digit_speaker_..)
              name = wav.stem.split("_")
              digit = int(name[0])
              speaker_id = sp.name
              items.append((str(wav), digit, speaker_id))
      return items

    def load_wav(self, path):
        wav, sr = torchaudio.load(path)  # [C, T]
        if sr != self.sr:
            wav = torchaudio.functional.resample(wav, sr, self.sr)
        wav = wav.mean(dim=0, keepdim=True)  # mono [1, T]
        return wav

    def __getitem__(self, idx):
        path, label, speaker = self.items[idx]
        wav = self.load_wav(path)

        # паддинг
        target_len = 16000
        current_len = wav.shape[1]
        if current_len < target_len:
            wav = F.pad(wav, (0, target_len - current_len))
        elif current_len > target_len:
            wav = wav[:, :target_len]

        # аугментация (на CPU)
        if self.augmentation:
            wav = wav.unsqueeze(0)  # [1, C, T]
            wav = fade_in_out(wav)
            wav = timemasking(wav, 1)
            wav = wav.squeeze(0)

        # расчет спектрограммы (на CPU)
        spec = mel_transform(wav)

        return spec, wav, label


def collate(batch):
    specs, wavs, labels = zip(*batch)

    # склеиваем тензоры в батч
    specs = torch.stack(specs)
    wavs = torch.stack(wavs)

    labels = torch.tensor(labels)

    return specs, wavs, labels



    # def __getitem__(self, idx):
    #     path, label, speaker = self.items[idx]
    #     wav = self.load_wav(path)

    #     # паддинг
    #     target_len = 16000
    #     current_len = wav.shape[1]
    #     if current_len < target_len:
    #         wav = F.pad(wav, (0, target_len - current_len))
    #     elif current_len > target_len:
    #         wav = wav[:, :target_len]

    #     if self.augmentation:
    #         wav = wav.unsqueeze(0)
    #         wav = fade_in_out(wav)
    #         wav = timemasking(wav, 1) # bs = 1
    #         wav = wav.squeeze(0)

    #     return wav, label

# def collate(batch):
#     wavs, labels = zip(*batch)
#     # собираем тензоры в батч
#     wavs = torch.stack(wavs)
#     labels = torch.tensor(labels, dtype=torch.long)

#     return wavs, labels


##### Создадим **конфиг. словарь**


In [5]:
hyperparameters = {
        'LR': 3e-4,
        'WEIGHT_DECAY': 1e-6,
        'B1':0.9,
        'B2':0.999,
        'EPOCHS': 5,
        'BATCH_SIZE': 128,
        'IMG_CHANNEL': 1,
        'CLASSES': 10,
        'EVAL_BATCH':128,
        'EVAL_EPOCHS':1,
        'MODEL_TITLE': 'contrastive'
}
config = DictConfig(hyperparameters)

##### Объявим **Мел. спек. transform**

In [6]:
class LogMelSpectrogram(T.MelSpectrogram):
    def __init__(self, eps=1e-8, **kwargs):
        super().__init__(**kwargs)
        self.eps = eps

    def forward(self, waveform):
        return (super().forward(waveform) + self.eps).log()

#Define Melspectogram (перенесем в getitem)
mel_transform = LogMelSpectrogram(
    sample_rate=16000,
    n_fft=2048,
    hop_length=128,
    n_mels=128,
    f_min=40,
    f_max=8000,
    mel_scale="slaney"
)

##### Сделаем **Split** данных
Split into train and eval by speaker: out of 60 speakers in total, 12 are assigned to the eval set.

In [7]:
def split_indices_by_speaker(dataset: AudioMNISTDataset, test_speakers: set):
    train_idxs = []
    test_idxs = []
    for idx, (_, _, spk) in enumerate(dataset.items):
        if spk in test_speakers:
            test_idxs.append(idx)
        else:
            train_idxs.append(idx)
    return train_idxs, test_idxs

In [8]:
# Split dataset by speakers
NUM_TEST_SPEAKERS = 12


# создаем датасет без аугментаций с  ними
full_WO_aug_t1 = AudioMNISTDataset(root=root, augmentation = False)
full_W_aug_t1 = AudioMNISTDataset(root=root, augmentation = True)

# train_transform = AudioAugmentations(mel_transform, train=True)
# valid_transform = AudioAugmentations(mel_transform, train=False)

# разделяем индексы
all_speakers = sorted({spk for (_, _, spk) in full_WO_aug_t1.items})
valid_speakers = set(all_speakers[-NUM_TEST_SPEAKERS:])
train_idxs_t1, valid_idxs_t1 = split_indices_by_speaker(full_WO_aug_t1, valid_speakers)

# создаем сабсеты
train_t1 = Subset(full_W_aug_t1, train_idxs_t1)
valid_t1 = Subset(full_WO_aug_t1, valid_idxs_t1)

# # DataLoaders (попробуем оптимизировать)
# train_dataloader_t1 = DataLoader(train_t1, batch_size=config.BATCH_SIZE, shuffle=True, collate_fn=collate, num_workers=2, pin_memory=True)
# valid_dataloader_t1 = DataLoader(valid_t1, batch_size=config.EVAL_BATCH, shuffle=False, collate_fn=collate, num_workers=2, pin_memory=True)


# Определим оптимальное количество воркеров
import multiprocessing
num_workers = min(4, multiprocessing.cpu_count())

train_dataloader = DataLoader(
    train_t1,
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    collate_fn=collate,
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=True  # экономит время на перезапуск воркеров между эпохами (PyTorch >= 1.7)
)

valid_dataloader = DataLoader(
    valid_t1,
    batch_size=config.EVAL_BATCH,
    shuffle=False,
    collate_fn=collate,
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=True
)


#### Используем энкодер **ResNet2D**


- **Input:** `[batch, channels, height, width]`  
- **First layer:** `Conv2d(7x7, stride=2)` → BatchNorm → ReLU → MaxPool2d  
- **Residual blocks:** `[2, 2, 2, 2]` blocks with channels `[64, 128, 256, 512]`  
- **Each block:** `Conv2d(1x1)` + `Conv2d(3x3)` with skip connection  
- **Output:** Global average pooled feature `[batch, 512, 1, 1]`  

This encoder *slides filters along time and frequency dimensions* to capture **spectro-temporal patterns** in spectrograms.

In [9]:
class block_resnet2d(nn.Module):
    def __init__(self, in_channels, out_channels, identity_downsample=None,stride=1):
        super(block_resnet2d, self).__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1, padding=0)
        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.identity_downsample = identity_downsample
        self.relu = nn.ReLU()


    def forward(self, x):
        identity = x

        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)

        x = self.conv2(x)
        x = self.bn2(x)

        if self.identity_downsample is not None:
            identity = self.identity_downsample(identity)

        x += identity
        x = self.relu(x)

        return x

class ResNet(nn.Module):
    # Resnet 18 [2, 2, 2, 2]
    def __init__(self, block, image_channels):
        super(ResNet, self).__init__()
        # for resnet18
        layers = [2, 2, 2, 2]
        self.expansion = 1

        self.in_channels = 64
        self.conv1 = nn.Conv2d(image_channels, self.in_channels, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(self.in_channels)
        self.relu = nn.ReLU()
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)


        self.layer1 = self._make_layer(block, layers[0], 64, stride=1)
        self.layer2 = self._make_layer(block, layers[1], 128, stride=2)
        self.layer3 = self._make_layer(block, layers[2], 256, stride=2)
        self.layer4 = self._make_layer(block, layers[3], 512, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1,1))


    def forward(self,x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)

        return x



    def _make_layer(self, block, num_residual_block, out_channels, stride):
        identity_downsample = None
        layers = []

        if stride != 1:
            identity_downsample = nn.Sequential(nn.Conv2d(self.in_channels,
                                                out_channels*self.expansion,
                                                kernel_size=1,
                                                stride=stride,
                                                bias=False),
                                                nn.BatchNorm2d(out_channels*self.expansion),
                                                )
        layers.append(
            block(self.in_channels,out_channels, identity_downsample, stride)
        )
        self.in_channels = out_channels * self.expansion

        for i in range(1, num_residual_block):
            layers.append(block(self.in_channels,out_channels ))

        return nn.Sequential(*layers)



def CreateResNet2D(img_channels=3):
    return ResNet(block_resnet2d, image_channels=img_channels)

#### Создадим **классификатор**

In [10]:
class Simple_classifier(nn.Module):
    """
    Linear classifier for AudioMNIST
    """
    def __init__(self, num_classes = 10, img_channels=1):
        super(Simple_classifier, self).__init__()

        self.output = nn.Sequential(
            CreateResNet2D(img_channels=img_channels),
            nn.Flatten(),
            nn.Linear(512, num_classes)
        )

    def forward(self,x):
        x = self.output(x)
        return x

classifier = Simple_classifier(num_classes = config.CLASSES)

#### Определим **аугментации**




In [11]:
def pitchshift(audio, SAMPLE_RATE=16000, shift = 2):
    """
    Pitch Shift (PS): randomly raises or lowers the pitch of the audio signal.\n
    Based on experimental observation,we found the range of pitch shifts that main-tained\n
    the overall coherency of the input audio was in the range [-15, 15] semitones.

    Attributes:
    - :param audio: audio tensor
    - :param SAMPLE_RATE: Sample rate, default=16000
    - :param shift: Pitch shift
    - :return: describe what it returns
    """
    assert audio != None, "audio should not be None"
    transform = transforms.PitchShift(sample_rate=SAMPLE_RATE, n_steps=shift)
    waveform_shift = transform(audio)
    return waveform_shift


def fade_in_out(audio):
    """
    Fade in/out (FD): gradually increases/decreases the intensity of the audio in the\n
    beginning/end of the audio signal.\n
    The degree of the fade was either linear, logarithmic or exponential (applied\n
    with uniform probability of 1/3). The size of the fade for either side of the\n
    audio signal could at maximum reach half of the audio signal. The size of the\n
    fade was another random parameter picked for each sample.
    """
    assert audio != None, "audio should not be None"
    _fade_shape = ['linear', 'logarithmic', 'exponential']

    # неэффективно
    # _fade_size = [i for i in range(1, int(audio.shape[2]/2))]
    # transform = transforms.Fade(fade_in_len=random.choice(_fade_size), fade_out_len=random.choice(_fade_size), fade_shape=random.choice(_fade_shape))

    max_fade_len = int(audio.shape[2] / 2)
    fade_in_len = random.randint(1, max_fade_len)
    fade_out_len = random.randint(1, max_fade_len)

    transform = transforms.Fade(fade_in_len=fade_in_len, fade_out_len=fade_out_len, fade_shape=random.choice(_fade_shape))
    waveform_fade_in_out = transform(audio)
    return waveform_fade_in_out

def add_white_noise_(signal, noise_level):
    """
    Noise Injection: mix the audio signal with random white, brown and pink noise.\n
    In our implementation, the intensity of the noise signal was randomly selected based\n
    on the strength of signal-to-noise ratio. Applied white, brown, or pink depending\n
    on an additional random parameter sampled from uniform distribution (Mixed Noise).
    """
    noise = torch.randn_like(signal)*torch.std(signal) * noise_level
    noisy_signal = signal + noise
    return noisy_signal


def timemasking(waveform, batch_size, sample_rate=16000):
    """
    Time masking:given an audio signal, in this transformation we randomly select a small\n
    segment of the full signal and set the signal values in that segment to normal noise or a\n
    constant value. In our implementation, we not only randomly selected the location of the\n
    masked segment but also we randomly selected the size of the segment. The size of the \n
    masked segment was set to maximally be 1/8 of the input signal.
    """
    """max_mask = int(sample_rate/8)*torch.ones(size=[batch_size])
    pos_iniziale = torch.randint(low=0, high=sample_rate, size=[batch_size])
    min_mask = sample_rate-pos_iniziale
    min_elements = torch.min(min_mask,max_mask)
    pos_finale = pos_iniziale+min_elements.to(torch.int)
    indices = torch.arange(sample_rate).unsqueeze(0).expand(batch_size, -1)
    range_mask = (indices >= pos_iniziale.unsqueeze(1)) & (indices <= pos_finale.unsqueeze(1))
    range_mask = range_mask[:,None,:]
    signal_[range_mask] = 0

    return signal_"""
    bs, ch, length = waveform.shape
    mask_len = length // 8
    augmented = waveform.clone()
    for i in range(bs):
        start = random.randint(0, length - mask_len)
        augmented[i, :, start:start + mask_len] = 0.0
    return augmented

def timeStretching():
    """
    slows down or speeds up the audio sample (while keeping the pitch unchanged).
    In this approach we transformed the signal by first computing the STFT of the signal, stretching
    it using a phase vocoder, and computing the inverse STFT to reconstruct the time domain signal.
    Following those transformations, we down-sampled or cropped the signal to match the same number
    of samples as the input signal. When the rate of stretching was greater than 1, the signal was
    sped up. Otherwise when the rate of stretching was less than 1, then the signal was slowed down.
    The rate of time stretching was randomized for each audio with range values of [0.5, 1.5].
    """
    return

#### Объявим функции
- createModelInput
- train_eval_loop
- epoch_loop



##### **createModelInput**

In [12]:
def createModelInput(audio, mel_transform, augmentation=True):

    audio = audio.unsqueeze(1)

    # CALCUALTE AUGMENTATION 1 AND AUGMENTATION 2
    if augmentation == True:
        audio = fade_in_out(audio)
        audio = timemasking(audio,audio.shape[0])

    # Create the augmented spectograms size [BATCH_SIZE, 3, 200, 200]
    #spectograms = createSpectograms(audio, mel_transform)
    spectograms = mel_transform(audio)
    spectograms = spectograms.to(device)

    return  spectograms, audio

##### **train_eval_loop**

In [13]:
def train_eval_loop(model, dataloader, train=True, optimizer = None, show_progress=False):
    model.to(device)
    if train:
        model.train()
    else:
        model.eval()

    # standard loss
    criterion = torch.nn.CrossEntropyLoss()
    correct = 0
    total = 0
    losses = []

    iterator = tqdm(dataloader, unit='step', leave=False) if show_progress else dataloader
    batch_size = dataloader.batch_size

    for specs, _, labels in iterator:

        # отправляем на GPU
        # audio = audio.to(device)
        specs = specs.to(device)
        labels = labels.to(device)

        # аугментации  и cпеки перенесем в __getitem__
        # чтобы не гонять данные туда сюда между CPU и GPU
        # if train:
        #     audio = fade_in_out(audio)
        #     audio = timemasking(audio, batch_size)
        # specs = mel_transform(audio)

        outputs = model(specs)
        loss = criterion(outputs, labels)

        # for train calc loss and backward
        if train:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        losses.append(loss.item())

        _, predicted = torch.max(outputs, dim=1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    accuracy = correct / total
    avg_loss = sum(losses) / len(losses)

    return accuracy, avg_loss


##### **epoch_loop**

In [14]:
def epoch_loop(model, trainloader, validloader):

    # define optimizer
    optimizer = optim.Adam(model.parameters(), lr=config.LR)

    history = {
        'train_loss': [], 'train_acc': [],
        'valid_loss': [], 'valid_acc': []
    }

    for epoch in range(config.EPOCHS):

        # train stage
        train_accuracy, train_loss = train_eval_loop(
            model,
            trainloader,
            train=True,
            optimizer=optimizer,
            show_progress=True
        )

        # eval stage
        eval_accuracy, eval_loss = train_eval_loop(
            model,
            validloader,
            train=False
        )

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_accuracy)
        history['valid_loss'].append(eval_loss)
        history['valid_acc'].append(eval_accuracy)

        print(f"Epoch {epoch+1}/{config.EPOCHS} - "
              f"Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.4f} | "
              f"Valid Loss: {eval_loss:.4f}, Valid Acc: {eval_accuracy:.4f}")

    return history


# epoch_loop(classifier, train_dataloader, valid_dataloader)

#### Обучим классификатор

In [15]:
# history_t1 = epoch_loop(classifier, train_dataloader, valid_dataloader)

# # лучший результат
# best_valid_acc_t1 = max(history_t1['valid_acc'])
# print(f"\nBest Supervised Validation Accuracy: {best_valid_acc_t1:.4f}")

⚠️ **Соединение с сервером Colab было сброшено, но результат был получен. Привел его ниже. При желании можно раскомментировать предыдущую ячейку, перезапустить все и проверить**

In [16]:


recovered_result = """
Epoch 1/5 - Train Loss: 0.2029, Train Acc: 0.9413 | Valid Loss: 0.1334, Valid Acc: 0.9640
Epoch 2/5 - Train Loss: 0.0235, Train Acc: 0.9945 | Valid Loss: 0.0999, Valid Acc: 0.9732
Epoch 3/5 - Train Loss: 0.0131, Train Acc: 0.9969 | Valid Loss: 0.1046, Valid Acc: 0.9662
Epoch 4/5 - Train Loss: 0.0102, Train Acc: 0.9975 | Valid Loss: 0.0713, Valid Acc: 0.9792
Epoch 5/5 - Train Loss: 0.0096, Train Acc: 0.9971 | Valid Loss: 0.1492, Valid Acc: 0.9495

Best Supervised Validation Accuracy: 0.9792
"""

print(recovered_result)


Epoch 1/5 - Train Loss: 0.2029, Train Acc: 0.9413 | Valid Loss: 0.1334, Valid Acc: 0.9640
Epoch 2/5 - Train Loss: 0.0235, Train Acc: 0.9945 | Valid Loss: 0.0999, Valid Acc: 0.9732
Epoch 3/5 - Train Loss: 0.0131, Train Acc: 0.9969 | Valid Loss: 0.1046, Valid Acc: 0.9662
Epoch 4/5 - Train Loss: 0.0102, Train Acc: 0.9975 | Valid Loss: 0.0713, Valid Acc: 0.9792
Epoch 5/5 - Train Loss: 0.0096, Train Acc: 0.9971 | Valid Loss: 0.1492, Valid Acc: 0.9495

Best Supervised Validation Accuracy: 0.9792



## Task 2(2 points)

Train the Multi-Format Contrastive Learning of Audio Representations model (implemented during the seminar) until convergence. Plot the loss and accuracy curves to verify that the training process has stabilized.

#### **Скопируем из ноутбука семинара недостающие блоки:**
- Энкодер ResNet1D.
- Основную модель Net.
- Классификатор EvaluationHead.
- Класс ContrastiveLoss.
- Функции create, train и evaluationphase.


#### Энкодер **ResNet1D**


##### **class block**

In [17]:
class block(nn.Module):
    def __init__(self, in_channels, out_channels, identity_downsample=None,stride=1):
        super(block, self).__init__()

        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=1, padding=0)
        self.bn1 = nn.BatchNorm1d(out_channels)

        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn2 = nn.BatchNorm1d(out_channels)

        self.identity_downsample = identity_downsample
        self.relu = nn.ReLU()


    def forward(self, x):
        identity = x

        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)

        x = self.conv2(x)
        x = self.bn2(x)

        if self.identity_downsample is not None:
            identity = self.identity_downsample(identity)

        x += identity
        x = self.relu(x)

        return x


##### **class ResNet1D**

In [18]:
class ResNet1D(nn.Module):
    # Resnet 18 [2, 2, 2, 2]
    def __init__(self, block):
        super(ResNet1D, self).__init__()
        # for resnet18
        layers = [2, 2, 2, 2]
        self.expansion = 1

        self.in_channels = 64
        self.conv1 = nn.Conv1d(1, self.in_channels, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(self.in_channels)
        self.relu = nn.ReLU()
        self.maxpool = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)


        self.layer1 = self._make_layer(block, layers[0], 64, stride=1)
        self.layer2 = self._make_layer(block, layers[1], 128, stride=2)
        self.layer3 = self._make_layer(block, layers[2], 256, stride=2)
        self.layer4 = self._make_layer(block, layers[3], 512, stride=2)

        self.avgpool = nn.AdaptiveAvgPool1d(output_size=1)


        # size after avgpool = [32, 512, 1]

    def forward(self,x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)

        #x = x.reshape(x.shape[0], -1)
        #x = self.fc(x)

        return x



    def _make_layer(self, block, num_residual_block, out_channels, stride):
        identity_downsample = None
        layers = []

        if stride != 1:
            identity_downsample = nn.Sequential(nn.Conv1d(self.in_channels,
                                                out_channels*self.expansion,
                                                kernel_size=1,
                                                stride=stride,
                                                bias=False),
                                                nn.BatchNorm1d(out_channels*self.expansion),
                                                )
        layers.append(
            block(self.in_channels,out_channels, identity_downsample, stride)
        )
        self.in_channels = out_channels * self.expansion

        for i in range(1, num_residual_block):
            layers.append(block(self.in_channels,out_channels ))

        return nn.Sequential(*layers)



def CreateResNet1D():
    return ResNet1D(block)


#### Основная модель **Net**

In [19]:
class Net(nn.Module):

    def __init__(self, img_channels = 1, num_classes = 35):
        super(Net, self).__init__()

        self.resnet_1D = CreateResNet1D()
        self.resnet_2D = CreateResNet2D(img_channels=img_channels)

        self.projection_head_1d = nn.Sequential(
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Linear(256, 128)
        )

        self.projection_head_2d = nn.Sequential(
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Linear(256, 128)
        )



    def forward(self, input_spectogram, input_audio):
        """
            resnet2d and resnet1d output is [BS, 512, 1, 1]
            Output:
                - audio_emb, specs_emb used for contrastive loss
                - audio, spectograms used for Evaluation layer
                - output used for semi supervised - cross entropy
        """

        audio = self.resnet_1D(input_audio)
        audio = audio.squeeze()
        spectograms = self.resnet_2D(input_spectogram)
        spectograms = spectograms.squeeze()

        audio_emb = self.projection_head_1d(audio)
        specs_emb = self.projection_head_2d(spectograms)

        return audio_emb, specs_emb, audio, spectograms



#### Классификатор **EvaluationHead**


In [20]:
class EvaluationHead(nn.Module):
    """
    Linear classifier head
    """
    def __init__(self, num_classes = 35):
        super(EvaluationHead, self).__init__()

        self.evaluation = nn.Sequential(
                    nn.Linear(512,num_classes)
        )

    def forward(self,x):
        x = self.evaluation(x)
        return x


#### Класс **ContrastiveLoss**

In [21]:
def device_as(t1, t2):
   return t1.to(t2.device)

class ContrastiveLoss(nn.Module):
  def __init__(self, temperature=0.5):
    super().__init__()
    self.temperature = temperature

  def forward(self, proj_1, proj_2):
    batch_size = proj_1.shape[0]

    # нормализуем и объединяем
    z_i = F.normalize(proj_1, p=2, dim=1)
    z_j = F.normalize(proj_2, p=2, dim=1)
    reps = torch.cat([z_i, z_j], dim=0)

    # косинусное сходство
    sim_matrix = F.cosine_similarity(reps.unsqueeze(1), reps.unsqueeze(0), dim=2)

    # позитивные пары
    sim_ij = torch.diag(sim_matrix, batch_size)
    sim_ji = torch.diag(sim_matrix, -batch_size)
    positives = torch.cat([sim_ij, sim_ji], dim=0)

    # маска для удаления диагонали
    mask = torch.eye(batch_size * 2, dtype=torch.bool).to(sim_matrix.device)

    # негативные пары (все, кроме диагонали)
    negatives = sim_matrix[~mask].view(batch_size * 2, -1)

    # собираем логиты [позитивная_пара, негатив_1, негатив_2, ...]
    logits = torch.cat([positives.unsqueeze(1), negatives], dim=1)
    logits /= self.temperature

    # целевые метки: всегда 0
    labels = torch.zeros(batch_size * 2).long().to(sim_matrix.device)

    # используем стандартный CrossEntropyLoss
    loss_fn = nn.CrossEntropyLoss()
    return loss_fn(logits, labels)


  # def calc_similarity_batch(self, a, b):
  #   rep = torch.cat([a,b])
  #   return F.cosine_similarity(rep.unsqueeze(1), rep.unsqueeze(0), dim=2)

  # def forward(self, proj_1, proj_2):
  #   batch_size = proj_1.shape[0]
  #   z_i = F.normalize(proj_1, p=2, dim=1)
  #   z_j = F.normalize(proj_2, p=2, dim=1)

  #   similarity_matrix = self.calc_similarity_batch(z_i, z_j)

  #   #######
  #   #aa#ab#
  #   #######
  #   #ba#bb#
  #   #######

  #   sim_ij = torch.diag(similarity_matrix, batch_size)
  #   sim_ji = torch.diag(similarity_matrix, -batch_size)

  #   positives = torch.cat([sim_ij, sim_ji])

  #   nominator = torch.exp(positives / self.temperature)

  #   mask = (~torch.eye(batch_size*2, batch_size*2).bool()).float()
  #   mask = device_as(mask, similarity_matrix)

  #   denominator = mask * torch.exp(similarity_matrix / self.temperature)
  #   all_losses = -torch.log(nominator / torch.sum(denominator, dim=1))
  #   loss = torch.sum(all_losses) / (2*batch_size)

  #   return loss

#### Функции **create**, **train** и **evaluationphase**

##### **create**


In [22]:

def create(config):

  # Create model
  model = Net(img_channels=config.IMG_CHANNEL, num_classes = config.CLASSES).to(device)

  # Define the constrastive loss
  loss = ContrastiveLoss()

  # болше не надо
  #Define Melspectogram and STFT (Magnitude and Phase)
  #  mel_transform = LogMelSpectrogram(sample_rate=16000, n_fft=2048, hop_length=128, n_mels=128,f_min=40, f_max=8000, mel_scale="slaney").to(device)

  # Define the optimizer, the paper use
  optimizer = optim.Adam(model.parameters(), lr=config.LR, betas=(config.B1, config.B2), weight_decay=config.WEIGHT_DECAY)

  return model, loss, optimizer

##### **train**


In [23]:
def train(model, closs, optimizer, trainloader, valloader, config):

    scaler = torch.amp.GradScaler()

    # сохраняем метрики:
    # контрастивный лосс - насколько хорошо модель учится сопоставлять позитивные пары
    # точность на валидации - насколько хорошо "замороженный" энкодер решает downstream-задачу
    train_losses, val_accuracies = [], []

    for epoch in range(config.EPOCHS):
        progress_bar = tqdm(total=len(trainloader), unit='step')
        losses = []
        clos_ = []

        for specs, wavs, labels in trainloader:
            # переносим на GPU
            specs = specs.to(device)
            wavs = wavs.to(device)

            # подаем в модель
            with torch.amp.autocast(device_type=str(device)):
                audio_emb, spect_emb, _, _ = model(specs, wavs)
                loss = closs(audio_emb, spect_emb)
                clos_.append(loss.item())

            # Calculate loss and backward
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            #progress bar stuff
            progress_bar.set_description(f"Epoch {epoch+1}/{config.EPOCHS}")
            progress_bar.set_postfix(loss=loss.item())  # Update the loss value
            progress_bar.update(1)

            # save loss to statistics
            losses.append(loss.item())

        # end for batch

        accuracy_test = evaluationphase(model, trainloader, valloader, config)
        train_losses.append(np.mean(losses))
        val_accuracies.append(accuracy_test)

        if epoch%3==0:
            # EVALUATION HEAD
            torch.save(model.state_dict(), f"models/model_{config.MODEL_TITLE}.pt")

    return train_losses, val_accuracies

##### **evaluationphase**


In [24]:
def evaluationphase(model, trainloader, val_loader, config):

    model.eval()
    # Get dataloaders
    # Freeze the gradients for model1
    for param in model.parameters():
        param.requires_grad = False

    def train_head(model, trainloader):

        # Create model
        modelEvaluation = None
        modelEvaluation = EvaluationHead(num_classes = config.CLASSES).to(device)

        # Define the optimizer, the paper use
        optimizer = optim.Adam(modelEvaluation.parameters(), lr=config.LR)

        criterion = torch.nn.CrossEntropyLoss()

        for epoch in range(config.EVAL_EPOCHS):
            progress_bar = tqdm(total=len(trainloader), unit='step', leave=False)
            losses = []
            for specs, wavs, labels in trainloader:
                # переносим на GPU
                specs = specs.to(device)
                wavs = wavs.to(device)
                labels = labels.to(device)

                # подаем в модель
                with torch.no_grad():
                    _, _, frozen_audio, frozen_spects = model(specs, wavs)

                labels_cat = torch.cat([labels, labels], dim = 0).to(device)

                inputs = torch.cat([frozen_audio, frozen_spects], dim = 0)
                outputs = modelEvaluation(inputs)
                loss = criterion(outputs, labels_cat)

                # Calculate loss and backward
                loss.backward()
                optimizer.step()

                losses.append(loss.item())

                #progress bar stuff
                progress_bar.set_description(f"Head tuning epoch {epoch+1}/{config.EVAL_EPOCHS}")
                progress_bar.set_postfix(loss=np.mean(losses))  # Update the loss value
                progress_bar.update(1)

            # end for batch


        return modelEvaluation

    def evaluation(model, model_eval, dataloader):
        model_eval.eval()
        progress_bar = tqdm(total=len(dataloader), unit='step')
        total = 0
        correct = 0
        with torch.no_grad():
            for i, (specs, wavs, labels) in enumerate(dataloader):
                # переносим на GPU
                specs = specs.to(device)
                wavs = wavs.to(device)
                labels = labels.to(device)

                labels_cat = torch.cat([labels, labels], dim = 0).to(device)

                # подаем в модель
                _, _, frozen_audio, frozen_spects = model(specs, wavs)


                inputs = torch.cat([frozen_audio, frozen_spects], dim = 0)
                outputs = model_eval(inputs)
                _, predicated = torch.max(outputs.data, 1)
                total += labels_cat.size(0)

                correct += (predicated == labels_cat).sum().item()

                #progress bar stuff
                progress_bar.set_description(f"Evaluation {i+1}/{len(dataloader)}")
                progress_bar.update(1)

            # end for batch

        return correct/total


    model_ = train_head(model, trainloader)
    accuracy_test = evaluation(model, model_, val_loader)
    print(f"Accuracy on validation: {accuracy_test}")
    model.train()
    for param in model.parameters():
        param.requires_grad = True

    return accuracy_test

#### Обучаем модель

In [25]:
hyperparameters_t2 = hyperparameters.copy()

hyperparameters_t2['LR'] = 3e-5

config_t2 = DictConfig(hyperparameters_t2)

In [26]:
# создадим папку для моделей
!mkdir -p models

In [ ]:
# создаем модель
model_task2, loss_fn_task2, optimizer_task2 = create(config_t2)

# обучение
train_losses_t2, val_accuracies_t2 = train(
    model_task2,
    loss_fn_task2,
    optimizer_task2,
    train_dataloader,
    valid_dataloader,
    config_t2
)



  0%|          | 0/188 [00:00<?, ?step/s]

  0%|          | 0/188 [00:00<?, ?step/s]

  0%|          | 0/47 [00:00<?, ?step/s]

Accuracy on validation: 0.16216666666666665


  0%|          | 0/188 [00:00<?, ?step/s]

  0%|          | 0/188 [00:00<?, ?step/s]

  0%|          | 0/47 [00:00<?, ?step/s]

Accuracy on validation: 0.21341666666666667


  0%|          | 0/188 [00:00<?, ?step/s]

#### **Визуализируем метрики**


In [ ]:

fig, ax = plt.subplots(1, 2, figsize=(15, 5))

# contr. loss
ax[0].plot(train_losses_t2, marker='o', linestyle='-')
ax[0].set_title("Contrastive Loss over Epochs")
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("InfoNCE Loss")
ax[0].grid(True)

# val acc
ax[1].plot(val_accuracies_t2, marker='o', linestyle='-', color='green')
ax[1].set_title("Validation Accuracy over Epochs")
ax[1].set_xlabel("Epoch")
ax[1].set_ylabel("Accuracy")
ax[1].grid(True)

fig.suptitle("Task 2: Contrastive Learning (InfoNCE) Training Metrics", fontsize=16)

plt.tight_layout()
plt.show()


## Task 3(4 points)

Replace the InfoNCE loss in Multi-Format Contrastive Learning of Audio Representations with a Non-Contrastive Learning method.
Train the model until convergence, then plot the loss and accuracy curves to check that training has stabilized.

You can try one of the following NCL methods:
- BYOL([paper](https://arxiv.org/pdf/2006.07733))
- SimSiam([paper](https://arxiv.org/pdf/2011.10566))
- Barlow Twins([paper](https://arxiv.org/pdf/2103.03230))
- VicReg([paper](https://arxiv.org/pdf/2105.04906))

Feel free to use a more recent Non-Contrastive approach if you prefer—just explain briefly why you chose it.


### ⚠️ Реализуем  **гибридный SimSiam**.
- В оригинальной статье предполагается использовать два одинаковых энкодера. Мы будем использовать тот же подход с двумя модальностями (волна, спектр) что и ранне в контрастивном обучении.
- За счет SimSiam (**predict** и **stop_gradient**) избавимся от необходимости негативных пар и коллапса.
- Благодаря разным представлениям данных на входах, модель получит больше информации данных, за счет чего должна лучше обучиться.

#### Создадим класс **SimSiamLoss**

Формула симметричного лосса из статьи:
$$
\mathcal{L} = \frac{1}{2} \mathcal{D}(p_1, z_2) + \frac{1}{2} \mathcal{D}(p_2, z_1)
$$
где $\mathcal{D}(p, z) = - \frac{p}{\|p\|_2} \cdot \frac{z}{\|z\|_2}$ (отрицательное косинусное сходство).

*   $\mathcal{D}(p_1, z_2)$ - predictor применяется к выходу левой ветки, а правая ветка "заморожена" (stop-gradient)
*   $\mathcal{D}(p_2, z_1)$ - зеркальная версия.


---

Используя симметричый лосс,  заставляем обе ветки учиться друг у друга.
- Левая ветка предсказывает правую.
- Правая - предсказывает левую.

In [ ]:
class SimSiamLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, p1, p2, z1, z2):

        loss1 = F.cosine_similarity(p1, z2).mean()
        loss2 = F.cosine_similarity(p2, z1).mean()

        return -0.5 * (loss1 + loss2)

#### Создадим **evaluationphase_simsiam**

In [ ]:
def evaluationphase_simsiam(model, val_loader, config):

    model.eval()
    # Get dataloaders
    # Freeze the gradients for model1
    for param in model.parameters():
        param.requires_grad = False

    def train_head(model, trainloader):

        # Create model
        modelEvaluation = None
        modelEvaluation = EvaluationHead(num_classes = config.CLASSES).to(device)

        # Define the optimizer, the paper use
        optimizer = optim.Adam(modelEvaluation.parameters(), lr=config.LR)

        criterion = torch.nn.CrossEntropyLoss()

        for epoch in range(config.EVAL_EPOCHS):
            progress_bar = tqdm(total=len(trainloader), unit='step', leave=False)
            losses = []

            for specs, wavs, labels in trainloader:
                # переносим на GPU
                specs = specs.to(device)
                wavs = wavs.to(device)
                labels = labels.to(device)

                # подаем в модель
                with torch.no_grad():
                    _, _, _, _, frozen_audio, frozen_spects = model(specs, wavs)

                labels_cat = torch.cat([labels, labels], dim = 0).to(device)

                inputs = torch.cat([frozen_audio, frozen_spects], dim = 0)
                outputs = modelEvaluation(inputs)
                loss = criterion(outputs, labels_cat)

                # Calculate loss and backward
                loss.backward()
                optimizer.step()

                losses.append(loss.item())

                #progress bar stuff
                progress_bar.set_description(f"Head tuning epoch {epoch+1}/{config.EVAL_EPOCHS}")
                progress_bar.set_postfix(loss=np.mean(losses))  # Update the loss value
                progress_bar.update(1)

            # end for batch


        return modelEvaluation

    def evaluation(model, model_eval, dataloader):
        model_eval.eval()
        progress_bar = tqdm(total=len(dataloader), unit='step')
        total = 0
        correct = 0
        with torch.no_grad():
            for i, (specs, wavs, labels) in enumerate(dataloader):

                # переносим на GPU
                specs = specs.to(device)
                wavs = wavs.to(device)
                labels = labels.to(device)

                _, _, _, _, frozen_audio, frozen_spects = model(specs, wavs)
                labels_cat = torch.cat([labels, labels], dim = 0).to(device)

                inputs = torch.cat([frozen_audio, frozen_spects], dim = 0)
                outputs = model_eval(inputs)
                _, predicated = torch.max(outputs.data, 1)
                total += labels_cat.size(0)

                correct += (predicated == labels_cat).sum().item()

                #progress bar stuff
                progress_bar.set_description(f"Evaluation {i+1}/{len(dataloader)}")
                progress_bar.update(1)

            # end for batch

        return correct/total


    model_ = train_head(model, train_loader)
    accuracy_test = evaluation(model, model_, val_loader)
    print(f"Accuracy on validation: {accuracy_test}")
    model.train()
    for param in model.parameters():
        param.requires_grad = True

    return accuracy_test

#### Создадим класс модели **SimSiamNet**


In [ ]:
class SimSiamNet(nn.Module):

    def __init__(self, img_channels = 1, num_classes = 35):
        super(SimSiamNet, self).__init__()

        self.resnet_1D = CreateResNet1D()
        self.resnet_2D = CreateResNet2D(img_channels=img_channels)

        self.output = nn.Sequential(
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Linear(256, 128)
        )

        self.predictor = nn.Sequential(
            nn.Linear(128, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Linear(256, 128)
        )


    def forward(self, input_spectogram, input_audio):
        """

        """

        # encoders
        audio = self.resnet_1D(input_audio).squeeze()
        spectograms = self.resnet_2D(input_spectogram).squeeze()

        # projection head embeddings
        audio_emb = self.output(audio)
        specs_emb = self.output(spectograms)

        # predictions
        p_audio = self.predictor(audio_emb)
        p_specs = self.predictor(specs_emb)

        # stop gradient
        sg_audio = audio_emb.detach()
        sg_specs = specs_emb.detach()

        return p_audio, p_specs, sg_audio, sg_specs, audio, spectograms

#### Объявим функцию **train_simsiam**


In [ ]:

def train_simsiam(model, closs, optimizer, trainloader, valloader, config, mel_transform, stft_trasform):

    scaler = torch.amp.GradScaler()

    # сохраняем метрики:
    # контрастивный лосс - насколько хорошо модель учится сопоставлять позитивные пары
    # точность на валидации - насколько хорошо "замороженный" энкодер решает downstream-задачу
    train_losses, val_accuracies = [], []

    for epoch in range(config.EPOCHS):
        progress_bar = tqdm(total=len(trainloader), unit='step')
        losses = []

        for specs, audio, labels in trainloader:
            # переносим на GPU
            specs = specs.to(device)
            audio = audio.to(device)

            with torch.amp.autocast(device_type=str(device)):
                p_audio, p_specs, sg_audio, sg_specs, _, _ = model(specs, audio)
                loss = closs(p_audio, p_specs, sg_audio, sg_specs)


            # Calculate loss and backward
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            #progress bar stuff
            progress_bar.set_description(f"Epoch {epoch+1}/{config.EPOCHS}")
            progress_bar.set_postfix(loss=loss.item())  # Update the loss value
            progress_bar.update(1)

            # save loss to statistics
            losses.append(loss.item())

        # end for batch

        accuracy_test = evaluationphase_simsiam(model, valloader, config, mel_transform, stft_trasform)
        train_losses.append(np.mean(losses))
        val_accuracies.append(accuracy_test)

        if epoch%3==0:
            # EVALUATION HEAD
            torch.save(model.state_dict(), f"models/model_{config.MODEL_TITLE}.pt")

    return train_losses, val_accuracies


#### Обучим **SimSiam**

In [ ]:
def create_simsiam(config):
    model = SimSiamNet(img_channels=1, num_classes=config.CLASSES).to(device)
    loss_fn = SimSiamLoss()

    return model, loss_fn

# создаем модель и оптимизатор
model_simsiam, loss_fn = create_simsiam(config)
optimizer_simsiam = torch.optim.Adam(model_simsiam.parameters(), lr=config.LR)

train_losses_t3, val_accuracies_t3 = train_simsiam(model_simsiam,
    loss_fn,
    optimizer_simsiam,
    train_dataloader,
    valid_dataloader,
    config
)



#### **Визуализируем метрики**


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))

# train loss
ax[0].plot(train_losses_t3, marker='o', linestyle='-')
ax[0].set_title("SimSiam Training Loss over Epochs")
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("SimSiam Loss")
ax[0].grid(True)

# val. acc.
ax[1].plot(val_accuracies_t3, marker='o', linestyle='-', color='green')
ax[1].set_title("SimSiam Validation Accuracy over Epochs")
ax[1].set_xlabel("Epoch")
ax[1].set_ylabel("Accuracy")
ax[1].grid(True)

fig.suptitle("SimSiam Training Metrics", fontsize=16)

plt.tight_layout()
plt.show()

## Task 4(2 points)

Evaluate and compare three training setups on the validation subset:

1. Supervised training

2. InfoNCE (contrastive learning)

3. Non-Contrastive Learning (NCL)

Use either the 2D or 1D encoder—or combine their embeddings.
Identify which setup performs best and explain why it outperforms the others.

In [ ]:
## Your code here